# UWCT Tritonia background-contrast images (bgc1) and draft dataset v9 (D-045) -- CPU, no training

**What this does:** preprocesses every frame of frozen v7 with method bgc1 (`src/dataset_audit/tritonia_bgc.py`,
parameters fixed by D-045). The 3 channels are R = original, G = brighter than the day's per-pixel median background
(x4), B = CLAHE. It then writes the **draft** dataset v9 (v7's frames, split and boxes, new images) and prints the
dataset fingerprint v9's YOLO view will have once frozen. Nothing is frozen, trained or evaluated here. Test frames
are processed so v9 is complete, but they are never looked at or scored (CLAUDE.md rule 7; cell 8 previews train and
val only).

**How to run (about 10-20 min, ESTIMATE):**
1. Settings: Accelerator **None (CPU)**, **Internet on**. Add-ons -> Secrets -> `UWCT_GITHUB_TOKEN` (ticked for this notebook).
2. **Add Input**, all four: `anusidhu7/uwct-tritonia-v7-inputs` (label source `tritonia_boxes_fixed2`),
   `anusidhu7/uwct-tritonia-v8-inputs` (`tritonia_label_batch2`: v7's frames plus the training-day background frames),
   `anusidhu7/uwct-tritonia-tracking-val` and `anusidhu7/uwct-tritonia-tracking-test` (every frame of the val / test days).
3. Cell 1: set `COMMIT` to the pushed commit of `anu/tritonia-preproc-size` that contains this notebook.
4. Run all cells. Cell 2 stops if an input is missing or laid out differently than expected. Cell 5 (plan only)
   stops if any frame has too few background frames.
5. Cell 9: download **`bgc1_outputs.zip`** before closing the session.

**After the session (on the PC, with Claude):**
1. Copy `v9/` from the zip into `data/processed/v9/` and check it.
2. Upload `tritonia_bgc1_v7/` together with `tritonia_boxes_fixed2/` as the private Kaggle dataset
   `anusidhu7/uwct-tritonia-v9-inputs`.
3. With explicit approval, freeze v9 (`scripts/freeze_dataset_version.py v9`).
4. Commit configs E053-E061 with the fingerprint printed in cell 7.
5. Write the GPU training notebook.

In [ ]:
# 1. Settings and helpers -- set COMMIT; keep everything else.
import os, re, subprocess, sys, json, shutil

REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "anu/tritonia-preproc-size"   # branch that holds COMMIT
COMMIT = "0000000000000000000000000000000000000000"  # REQUIRED: full sha, pushed
PARENT, VERSION, DECISION = "v7", "v9", "D-045"
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"

WORK = "/tmp"                          # scratch, not saved
REPO_DIR = f"{WORK}/uwct"
LOCAL_BOXES = f"{WORK}/raw/tritonia_boxes_fixed2"
W = "/kaggle/working"
OUT = f"{W}/tritonia_bgc1_v7"          # the processed batch (new folder)
LOGS = f"{W}/bgc1_logs"
os.makedirs(LOGS, exist_ok=True)

assert "KAGGLE_KERNEL_RUN_TYPE" in os.environ, "this notebook is for Kaggle"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT) and set(COMMIT) != {"0"}, "Set COMMIT to the exact 40-character commit sha to run"


def sh(*cmd, env=None):
    """Run a command in the repo, stream its output, raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")


def get_secret(name):
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret(name)


print("settings OK | output:", OUT)

In [ ]:
# 2. The four attached inputs (read-only under /kaggle/input; exactly one match each, else stop).
def find_dirs(dataset, pred):
    return sorted(d for d, subdirs, files in os.walk("/kaggle/input") if dataset in d and pred(d, subdirs, files))


def one(dataset, what, pred):
    hits = find_dirs(dataset, pred)
    assert len(hits) == 1, f"expected one {what} in {dataset} under /kaggle/input, found {hits} -- attach it (Add Input)"
    return hits[0]


SRC_BOXES = one("uwct-tritonia-v7-inputs", "tritonia_boxes_fixed2/",
                lambda d, s, f: os.path.basename(d) == "tritonia_boxes_fixed2" and "labels" in s)
SRC_BATCH = one("uwct-tritonia-v8-inputs", "tritonia_label_batch2/",
                lambda d, s, f: os.path.basename(d) == "tritonia_label_batch2" and "images" in s)
is_seq = lambda d, s, f: "images" in s and "manifest.csv" in f and "sequence.json" in f
SRC_VAL = one("uwct-tritonia-tracking-val", "sequence folder (images/, manifest.csv, sequence.json)", is_seq)
SRC_TEST = one("uwct-tritonia-tracking-test", "sequence folder (images/, manifest.csv, sequence.json)", is_seq)
for name, d in [("label source", SRC_BOXES), ("batch 2", SRC_BATCH), ("val sequence", SRC_VAL), ("test sequence", SRC_TEST)]:
    print(f"{name}: {d}")
for d in (SRC_VAL, SRC_TEST):  # days only, no frames are looked at
    print(os.path.basename(d), "days:", sorted(json.load(open(os.path.join(d, "sequence.json"))).get("days", {})))

In [ ]:
# 3. Check out the exact commit (read-only token from Kaggle Secrets, as in kaggle_tritonia_v8.ipynb cell 3:
#    the token is only in the environment of the one git process, never in a URL, file, .git/config or output).
import stat

REPO_URL = f"https://github.com/{REPO}.git"
ASKPASS = f"{WORK}/uwct_git_askpass.sh"
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\n'
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
            'esac\n')
os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    env = {**os.environ, "GIT_ASKPASS": ASKPASS, "GIT_TERMINAL_PROMPT": "0", "UWCT_GIT_TOKEN": get_secret(TOKEN_SECRET_NAME)}
    try:
        r = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd, env=env, capture_output=True, text=True)
        err = r.stderr.replace(env["UWCT_GIT_TOKEN"], "***")
        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = os.path.isfile(config) and env["UWCT_GIT_TOKEN"] in open(config).read()
    finally:
        env.clear()
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    assert not leaked, "token found in .git/config"


if not os.path.isdir(REPO_DIR):
    git_remote("clone", "--branch", BRANCH, REPO_URL, REPO_DIR, cwd=WORK)
else:
    git_remote("fetch", "origin", BRANCH)
git("checkout", "--detach", COMMIT)
assert git("rev-parse", "HEAD") == COMMIT
print("commit:", COMMIT, "| working tree:", git("status", "--porcelain") or "clean")
open(f"{LOGS}/code_version.txt", "w").write(f"{REPO_URL}\n{BRANCH}\n{COMMIT}\n")

In [ ]:
# 4. Environment (numpy + OpenCV come with Kaggle; nothing is installed) and the bgc1 unit tests,
#    including the pixel test that is skipped on the PC (no numpy/OpenCV there).
import numpy, cv2
versions = {"python": sys.version.split()[0], "numpy": numpy.__version__, "opencv": cv2.__version__}
print(versions)
json.dump(versions, open(f"{LOGS}/versions.json", "w"), indent=1)
# discover -s tests imports the file as a top-level module: Kaggle has another package named "tests" installed,
# which would shadow the repo's tests/ folder (no __init__.py) under the name tests.test_tritonia_bgc.
sh(sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_tritonia_bgc.py", "-v")

In [ ]:
# 5. Plan only: every v7 frame found byte-identical in batch 2 and enough background frames for each. Writes nothing.
SCRIPT_ARGS = ["--version", PARENT, "--batch", SRC_BATCH, "--sequence", SRC_VAL, "--sequence", SRC_TEST]
sh(sys.executable, "scripts/preprocess_tritonia_bgc.py", *SCRIPT_ARGS, "--out", OUT, "--plan-only")

In [ ]:
# 6. Preprocess the 150 v7 frames -> OUT (a new folder; refuses to overwrite). Per-split summary at the end.
#    Look at it before going on: neighbours_min >= 8 everywhere; mean_abs_offset_error is larger on train
#    (sparse batch-2 frames) than on val/test (every frame) -- expected, D-045 LIMITATION; mean_abs_diff_max far
#    above its median would point to a camera bump or lighting jump in some frame (check preprocess_manifest.csv).
sh(sys.executable, "scripts/preprocess_tritonia_bgc.py", *SCRIPT_ARGS, "--out", OUT)
print(json.dumps(json.load(open(f"{OUT}/preprocess_report.json"))["per_split"], indent=1))

In [ ]:
# 7. Draft v9 in the clone (data/processed/v9/, NOT frozen) and the fingerprint its YOLO view will have once frozen.
if os.path.isdir(LOCAL_BOXES):
    shutil.rmtree(LOCAL_BOXES)
shutil.copytree(SRC_BOXES, LOCAL_BOXES)
os.environ["UWCT_TRITONIA_BOXES_ROOT"] = LOCAL_BOXES  # a read-only raw root for every script
FP_SCRATCH = f"{WORK}/v9_fingerprint"
if os.path.isdir(FP_SCRATCH):
    shutil.rmtree(FP_SCRATCH)
sh(sys.executable, "scripts/make_tritonia_bgc_version.py", "--version", VERSION, "--decision", DECISION,
   "--parent", PARENT, "--processed", OUT)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from dataset_audit.tritonia_bgc import draft_fingerprint
from dataset_audit.versioning import read_version_meta
FP = draft_fingerprint(VERSION, LOCAL_BOXES, FP_SCRATCH)
meta = read_version_meta(VERSION)
assert meta["status"] == "draft" and meta["derived_from"] == PARENT, meta
open(f"{LOGS}/v9_fingerprint.txt", "w").write(FP + "\n")
print("v9 draft OK | dataset_fingerprint after freezing:", FP)

In [ ]:
# 8. Visual check: train and val frames only (never test, rule 7). Each preview = R / G / B channel side by side.
import csv
PREVIEW = f"{W}/bgc1_previews"
os.makedirs(PREVIEW, exist_ok=True)
rows = list(csv.DictReader(open(f"{OUT}/preprocess_manifest.csv")))
picked = [r for split in ("train", "val") for r in [x for x in rows if x["split"] == split][:3]]
for r in picked:
    img = cv2.imread(f"{OUT}/images/{r['filename']}", cv2.IMREAD_COLOR)  # stored B, G, R
    strip = numpy.vstack([img[:, :, 2], img[:, :, 1], img[:, :, 0]])  # original / contrast / CLAHE
    cv2.imwrite(f"{PREVIEW}/{r['split']}_{r['filename']}", strip)
    print(r["split"], r["filename"], "| neighbours", r["n_neighbours"], "| mean |frame - background|", r["mean_abs_diff"])
from IPython.display import Image, display
for name in sorted(os.listdir(PREVIEW)):
    print(name); display(Image(f"{PREVIEW}/{name}", width=700))

In [ ]:
# 9. One zip to download: the processed batch, the v9 draft files, logs and previews.
STAGE = f"{WORK}/bgc1_outputs"
if os.path.isdir(STAGE):
    shutil.rmtree(STAGE)
shutil.copytree(OUT, f"{STAGE}/tritonia_bgc1_v7")
shutil.copytree(os.path.join(REPO_DIR, "data", "processed", VERSION), f"{STAGE}/{VERSION}")
shutil.copytree(LOGS, f"{STAGE}/bgc1_logs")
shutil.copytree(PREVIEW, f"{STAGE}/bgc1_previews")
zip_path = shutil.make_archive(f"{W}/bgc1_outputs", "zip", STAGE)
import hashlib
digest = hashlib.sha256(open(zip_path, "rb").read()).hexdigest()
print(zip_path, os.path.getsize(zip_path), "bytes | sha256", digest)
from IPython.display import FileLink
display(FileLink(os.path.relpath(zip_path, W)))